In [29]:
# Notebook: Leaderboard Builder
#
# Rebuilds the leaderboard table from scratch by looping through every artist
# that has at least min_cnt ratings and inserting their single best song
# (highest piki_score).  The result is one row per artist in the leaderboard.

import warnings
warnings.filterwarnings('ignore')
import pandas as pd

# Connect to the museiq database and expose `connection`, `engine`, and `DB`
%run db.py

In [32]:
def delete_leaderboard():
    """Clear all rows from the leaderboard table."""
    DB.execute('DELETE FROM leaderboard')
    print("Leaderboard cleared.")


def get_artists(min_cnt=20):
    """Return all artist_ids that have at least one song with >= min_cnt total ratings."""
    sql = """
        SELECT ps.artist_id, MAX(a.name) AS artist,max(pop_decile) as pop_decile
        FROM piki_score ps
        INNER JOIN catalog c ON ps.song_id = c.id
        INNER JOIN artists a ON c.artist_id = a.id
        WHERE num_likes + num_superlikes + num_dislikes >= %s
        GROUP BY ps.artist_id
        ORDER BY pop_decile desc
    """ % min_cnt
    return DB.fetch(sql)


def get_best_song(artist_id, min_cnt=20):
    """Return the highest-piki_score song for an artist as a dict, or None."""
    sql = """
        SELECT ps.song_id, c.artist, c.title,
               num_likes + num_superlikes + num_dislikes AS cnt,
               ps.artist_id, ps.piki_score, ag.genre
        FROM piki_score ps
        INNER JOIN catalog c ON ps.song_id = c.id
        left join artist_genre ag on c.artist_id=ag.artist_id
        WHERE ps.artist_id = %s
          AND num_likes + num_superlikes + num_dislikes >= %s
        ORDER BY cnt DESC
        LIMIT 1
    """ % (artist_id, min_cnt)
    rows = DB.fetch(sql)
    return rows[0] if rows else None


def insert_into_leaderboard(song):
    """Insert a song dict into the leaderboard table."""
    artist = song['artist'].replace("'", "\\'")
    title  = song['title'].replace("'", "\\'")
    sql = """
        INSERT INTO leaderboard (song_id, artist, title, tag, cnt, artist_id, piki_score)
        VALUES (%s, '%s', '%s', '%s', %s, %s, %s)
    """ % (song['song_id'], artist, title,song['genre'], song['cnt'], song['artist_id'], song['piki_score'])
    DB.execute(sql)

In [33]:
min_cnt = 100

delete_leaderboard()

artists = get_artists(min_cnt)
print(f"Found {len(artists):,} artists to process")

inserted = 0
errors = 0
for artist in artists:
    try:
        song = get_best_song(artist['artist_id'], min_cnt)
        if song:
            insert_into_leaderboard(song)
            inserted += 1
    except Exception as e:
        errors += 1
        print(f"Error for {artist['artist']}: {e}")

print(f"Done — inserted {inserted:,} songs ({errors} errors)")

Leaderboard cleared.
Found 4,776 artists to process
Done — inserted 4,776 songs (0 errors)


In [34]:
# Preview the top 20 leaderboard entries by piki_score
rows = DB.fetch("SELECT * FROM leaderboard ORDER BY piki_score DESC LIMIT 20")
pd.DataFrame(rows)

,user_id,song_id,artist,title,tag,liked,cnt,artist_id,rank,piki_score,id
0,None,2020546,Moondog,Sandalwood,rock,None,211,1060,None,100.0,1397540
1,None,61972,La Femme,Antitaxi,indie pop,None,452,5672,None,100.0,1396312
2,None,8397050,ATB,Ecstasy (Official Video HD),trance,None,456,351,None,100.0,1395779
3,None,25993,Faithless,Insomnia,electronic,None,405,519,None,100.0,1396075
4,None,59850,Staind,Epiphany,rock,None,410,2624,None,100.0,1395723
5,None,13739,Steven Wilson,Drive Home,rock,None,405,2245,None,100.0,1397491
6,None,4192087,Jóhann Jóhannsson,A Sparrow Alighted Upon Our Shoulder,soundtrack,None,449,1390,None,100.0,1395679
7,None,351939,Tony Bennett,My Foolish Heart,jazz,None,370,368,None,100.0,1395266
8,None,5631500,Eva Weel Skram,Nå tennes tusen julelys,indie,None,229,133024,None,100.0,1397233
9,None,8104384,Mora,512,latin,None,433,673039,None,100.0,1394704
